In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bigdata-gbl-quiz"
PROCESSED_FILE = os.path.join(PROJECT_ROOT, "data", "processed", "quiz_features.csv")

df = pd.read_csv(PROCESSED_FILE)
print("Shape:", df.shape)
df.head()

Shape: (19991, 15)


,learner_id,question_id,topic,difficulty,time_spent_seconds,attempts_count,hints_used,correct,time_per_attempt,attempt_index,recent_success_5,recent_hints_5,recent_time_5,questions_completed,avg_difficulty_so_far
0,L0001,Q0103,Geography,2,73,3,2,0,24.333333,1,0.500000,0.0,56.000000,1,2.000000
1,L0001,Q0561,History,2,74,3,1,0,24.666667,2,0.000000,2.0,73.000000,2,2.000000
2,L0001,Q0189,ArtsCulture,1,37,2,0,1,18.500000,3,0.000000,1.5,73.500000,3,2.000000
3,L0001,Q0443,ArtsCulture,1,60,2,1,0,30.000000,4,0.333333,1.0,61.333333,4,1.666667
4,L0001,Q0392,Technology,1,33,2,0,1,16.500000,5,0.250000,1.0,61.000000,5,1.500000


**Choose features (X) and target (y)**

In [ ]:
target = "correct"

feature_cols = [
    "difficulty",
    "time_spent_seconds",
    "attempts_count",
    "hints_used",
    "time_per_attempt",
    "recent_success_5",
    "recent_hints_5",
    "recent_time_5",
    "avg_difficulty_so_far",
    "questions_completed",
    "topic"
]

df_model = df[["learner_id"] + feature_cols + [target]].copy()
df_model.head()

,learner_id,difficulty,time_spent_seconds,attempts_count,hints_used,time_per_attempt,recent_success_5,recent_hints_5,recent_time_5,avg_difficulty_so_far,questions_completed,topic,correct
0,L0001,2,73,3,2,24.333333,0.500000,0.0,56.000000,2.000000,1,Geography,0
1,L0001,2,74,3,1,24.666667,0.000000,2.0,73.000000,2.000000,2,History,0
2,L0001,1,37,2,0,18.500000,0.000000,1.5,73.500000,2.000000,3,ArtsCulture,1
3,L0001,1,60,2,1,30.000000,0.333333,1.0,61.333333,1.666667,4,ArtsCulture,0
4,L0001,1,33,2,0,16.500000,0.250000,1.0,61.000000,1.500000,5,Technology,1


**Split by learner**

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df_model[feature_cols]
y = df_model[target]
groups = df_model["learner_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Train learners:", groups.iloc[train_idx].nunique())
print("Test learners:", groups.iloc[test_idx].nunique())

Train rows: 15960
Test rows: 4031
Train learners: 240
Test learners: 60


**Check class balance**

In [ ]:
print("Train correct rate:", round(y_train.mean(), 3))
print("Test correct rate:", round(y_test.mean(), 3))

Train correct rate: 0.329
Test correct rate: 0.318


**Predictive Modelling (continue)**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = [c for c in feature_cols if c != "topic"]
categorical_features = ["topic"]

# For Logistic Regression: scale numeric + one-hot for topic
preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# For tree models: no scaling needed, but still one-hot topic
preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

**Model 1: Logistic Regression (baseline)**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

lr_model = Pipeline(steps=[
    ("preprocess", preprocess_lr),
    ("model", LogisticRegression(max_iter=300, class_weight="balanced"))
])

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nReport:\n", classification_report(y_test, y_pred_lr, digits=3))

=== Logistic Regression ===
ROC-AUC: 0.7
Confusion matrix:
 [[1875  874]
 [ 502  780]]

Report:
               precision    recall  f1-score   support

           0      0.789     0.682     0.732      2749
           1      0.472     0.608     0.531      1282

    accuracy                          0.659      4031
   macro avg      0.630     0.645     0.631      4031
weighted avg      0.688     0.659     0.668      4031



**Model 2: Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nReport:\n", classification_report(y_test, y_pred_rf, digits=3))

=== Random Forest ===
ROC-AUC: 0.66
Confusion matrix:
 [[2477  272]
 [ 966  316]]

Report:
               precision    recall  f1-score   support

           0      0.719     0.901     0.800      2749
           1      0.537     0.246     0.338      1282

    accuracy                          0.693      4031
   macro avg      0.628     0.574     0.569      4031
weighted avg      0.662     0.693     0.653      4031



**Model 3: Gradient Boosting**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", GradientBoostingClassifier(random_state=42))
])

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

print("=== Gradient Boosting ===")
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_gb), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("\nReport:\n", classification_report(y_test, y_pred_gb, digits=3))

=== Gradient Boosting ===
ROC-AUC: 0.694
Confusion matrix:
 [[2523  226]
 [ 979  303]]

Report:
               precision    recall  f1-score   support

           0      0.720     0.918     0.807      2749
           1      0.573     0.236     0.335      1282

    accuracy                          0.701      4031
   macro avg      0.647     0.577     0.571      4031
weighted avg      0.673     0.701     0.657      4031



**Compare all models in one table**

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_lr),
        "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
        "Recall": recall_score(y_test, y_pred_lr),
        "F1": f1_score(y_test, y_pred_lr),
        "ROC-AUC": roc_auc_score(y_test, y_prob_lr)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Precision": precision_score(y_test, y_pred_rf, zero_division=0),
        "Recall": recall_score(y_test, y_pred_rf),
        "F1": f1_score(y_test, y_pred_rf),
        "ROC-AUC": roc_auc_score(y_test, y_prob_rf)
    },
    {
        "Model": "Gradient Boosting",
        "Accuracy": accuracy_score(y_test, y_pred_gb),
        "Precision": precision_score(y_test, y_pred_gb, zero_division=0),
        "Recall": recall_score(y_test, y_pred_gb),
        "F1": f1_score(y_test, y_pred_gb),
        "ROC-AUC": roc_auc_score(y_test, y_prob_gb)
    }
]).round(3)

results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.659,0.472,0.608,0.531,0.700
1,Random Forest,0.693,0.537,0.246,0.338,0.660
2,Gradient Boosting,0.701,0.573,0.236,0.335,0.694


**Save it into report/**

In [11]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/bigdata-gbl-quiz"
REPORT_DIR = os.path.join(PROJECT_ROOT, "report")
os.makedirs(REPORT_DIR, exist_ok=True)

results_file = os.path.join(REPORT_DIR, "model_comparison_results.csv")
results.to_csv(results_file, index=False)

print("✅ Saved:", results_file)

✅ Saved: /content/drive/MyDrive/bigdata-gbl-quiz/report/model_comparison_results.csv
